# QENS — Quantum Entanglement Navigation Simulator

An interactive walkthrough of quantum-enhanced deep-space navigation.

**What you will see:**
1. Bell state generation and entanglement measurement (Qiskit)
2. How entanglement degrades over time (decoherence)
3. Quantum sensor sensitivity: Standard Quantum Limit vs Heisenberg limit
4. Navigation comparison: classical INS vs quantum INS
5. Kalman filter + X-ray pulsar fusion: three-way comparison

> Run cells top-to-bottom with **Shift+Enter**.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('Environment ready.')


## 1. Bell State Generation

A **Bell state** is a maximally entangled two-qubit state. The circuit is
simple: a Hadamard gate puts qubit 0 into superposition, a CNOT gate
entangles it with qubit 1.

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

Measuring one qubit instantly determines the other — regardless of distance.
This is the physical principle behind quantum sensor arrays.


In [ ]:
from qens.quantum.entanglement import EntangledPair, BellState

pair = EntangledPair(BellState.PHI_PLUS, shots=8192)
counts = pair.measure()
corr = pair.correlation()
fidelity = pair.entanglement_fidelity()

print(f'Bell state:       {pair.bell_state}')
print(f'Measurement counts: {counts}')
print(f'<Z x Z> correlation: {corr:.4f}  (ideal: +1.000)')
print(f'Fidelity:           {fidelity:.4f}  (1.000 = perfect)')


In [ ]:
# Visualise the measurement histogram
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(counts.keys(), counts.values(), color=['#2ecc71', '#3498db'])
ax.set_xlabel('Measurement outcome')
ax.set_ylabel('Counts')
ax.set_title(f'Bell state {pair.bell_state} — {pair.shots} shots')
plt.tight_layout(); plt.show()


## 2. Entanglement Decoherence

Real quantum systems interact with their environment and lose entanglement
over the **dephasing time T₂**. The correlation decays as:

$$\langle Z \otimes Z \rangle(t) = e^{-t/T_2}$$

For quantum navigation sensors, T₂ sets the maximum useful measurement
window before the entanglement advantage is lost.


In [ ]:
T2 = 1.0  # dephasing time [s] — trapped ions: ~1 s, superconducting: ~100 µs
t_vals = np.linspace(0, 3 * T2, 20)

simulated_corr = []
for t in t_vals:
    decohered = EntangledPair(BellState.PHI_PLUS, shots=2048).decohere(t, T2)
    simulated_corr.append(decohered.correlation())

t_smooth = np.linspace(0, 3 * T2, 300)
theory = np.exp(-t_smooth / T2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_smooth / T2, theory, 'b-', lw=2, label=r'Theory: $e^{-t/T_2}$')
ax.scatter(t_vals / T2, simulated_corr, color='red', zorder=5, label='Qiskit simulation')
ax.axhline(0, color='gray', linestyle=':')
ax.set_xlabel(r'$t \ / \ T_2$')
ax.set_ylabel(r'$\langle Z \otimes Z \rangle$')
ax.set_title(f'Entanglement Decoherence  (T2 = {T2} s)')
ax.legend(); plt.tight_layout(); plt.show()


## 3. Quantum Sensor Sensitivity

Atom interferometers measure rotation and acceleration by splitting a cold
atom cloud along two paths and measuring the phase difference at recombination.

| Regime | Sensitivity | Formula |
|--------|-------------|---------|
| Standard Quantum Limit (SQL) | 1/√N | unentangled atoms |
| **Heisenberg Limit** | **1/N** | **entangled NOON states** |

The entangled case gives **√N times better sensitivity** — 31.6× for N = 1000.


In [ ]:
from qens.quantum.gyroscope import QuantumGyroscope, GyroscopeParams

N_vals = np.logspace(1, 5, 200).astype(int)
sql_sens  = [QuantumGyroscope(GyroscopeParams(N=N, entangled=False)).sensitivity for N in N_vals]
heis_sens = [QuantumGyroscope(GyroscopeParams(N=N, entangled=True)).sensitivity  for N in N_vals]

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(N_vals, sql_sens,  color='#e74c3c', lw=2, label='SQL  —  1/√N')
ax.loglog(N_vals, heis_sens, color='#2ecc71', lw=2, label='Heisenberg  —  1/N')
ax.set_xlabel('Number of atoms N')
ax.set_ylabel('Gyroscope sensitivity [rad/s]')
ax.set_title('Quantum Sensing: SQL vs Heisenberg Limit')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

for N in [100, 1000, 10000]:
    g = QuantumGyroscope(GyroscopeParams(N=N, entangled=True))
    print(f'N={N:6d}:  Heisenberg = {g.sensitivity:.2e} rad/s  ({g.sensitivity_ratio_vs_classical():.1f}x better than SQL)')


## 4. Classical vs Quantum INS (10-minute LEO orbit)

Both navigators start with perfect initial conditions and integrate their
respective sensor measurements. The only difference is the noise floor.

The **reference trajectory** uses a zero-noise INS so that we measure only
sensor noise contribution — not numerical integration artefacts.


In [ ]:
from qens.trajectory.navigator import Navigator

nav = Navigator(seed=42)
result = nav.run(duration=600.0, dt=1.0)

t_min = result.times / 60

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.semilogy(t_min, result.classical_errors, color='#e74c3c', lw=1.5, label='Classical INS')
ax1.semilogy(t_min, result.quantum_errors,   color='#2ecc71', lw=1.5, label='Quantum INS (Heisenberg)')
ax1.set_xlabel('Time [min]'); ax1.set_ylabel('Position error [m]')
ax1.set_title('Navigation Error Growth'); ax1.legend(); ax1.grid(alpha=0.3)

ratio = result.classical_errors / np.maximum(result.quantum_errors, 1e-12)
ax2.plot(t_min, ratio, color='#3498db', lw=1.5)
ax2.axhline(1, color='gray', ls='--', alpha=0.5)
ax2.set_xlabel('Time [min]'); ax2.set_ylabel('Error ratio (classical / quantum)')
ax2.set_title('Quantum Advantage Factor'); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f'Classical INS final error: {result.classical_errors[-1]:.2f} m')
print(f'Quantum INS final error:   {result.quantum_errors[-1]:.4f} m')
print(f'Quantum advantage:         {result.classical_errors[-1]/result.quantum_errors[-1]:.1f}x')


## 5. Kalman Filter + X-ray Pulsar Navigation (1 hour)

In deep space, X-ray pulsar timing provides periodic position fixes.
An **error-state Kalman filter** fuses these with INS predictions optimally.

| Navigator | Sensor | Corrections |
|-----------|--------|-------------|
| Pure classical INS | Tactical MEMS | None |
| KF + classical | Tactical MEMS | Pulsar every 150 s |
| KF + quantum | Heisenberg limit | Pulsar every 150 s |

Pulsar system: 3 pulsars, ~200 m combined accuracy (near-future XNAV).


In [ ]:
from qens.trajectory.comparison import NavigationComparison
from qens.sources.pulsar import PulsarNavSystem

pulsar = PulsarNavSystem()
print(pulsar)

comp = NavigationComparison(N_atoms=1000, seed=42)
result_kf = comp.run(duration=3600.0, dt=1.0)

print(f'\nFixes received: {result_kf.n_fixes}')
print(f'Pure classical INS:  {result_kf.pure_classical_errors[-1]:>10.1f} m')
print(f'KF + classical:      {result_kf.kf_classical_errors[-1]:>10.1f} m')
print(f'KF + quantum:        {result_kf.kf_quantum_errors[-1]:>10.4f} m')


In [ ]:
t_min = result_kf.times / 60

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle('QENS — Kalman Filter + Pulsar Navigation (1 hour)', fontsize=13, fontweight='bold')

ax1.semilogy(t_min, result_kf.pure_classical_errors, color='#e74c3c', lw=1.5, label='Pure classical INS')
ax1.semilogy(t_min, result_kf.kf_classical_errors,   color='#e67e22', lw=1.8, label='KF + classical + pulsar')
ax1.semilogy(t_min, result_kf.kf_quantum_errors,     color='#2ecc71', lw=2.0, label='KF + quantum + pulsar')
ax1.axhline(result_kf.pulsar_accuracy, color='gray', ls='--', alpha=0.6, label=f'Pulsar accuracy ({result_kf.pulsar_accuracy:.0f} m)')
ax1.set_ylabel('Position error [m]'); ax1.set_title('Navigation Error')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

ax2.semilogy(t_min, result_kf.kf_classical_sigma, color='#e67e22', lw=1.8, ls='--', label='KF classical 1-sigma')
ax2.semilogy(t_min, result_kf.kf_quantum_sigma,   color='#2ecc71', lw=2.0, ls='--', label='KF quantum 1-sigma')
ax2.set_xlabel('Time [min]'); ax2.set_ylabel('Uncertainty [m]')
ax2.set_title('Kalman Filter Predicted Uncertainty')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()


## Summary

| Scenario | Error after simulation |
|----------|----------------------|
| Classical INS (10 min LEO) | ~5 m |
| Quantum INS (10 min LEO) | ~0.04 m — **124× better** |
| Pure classical INS (1 hour) | ~41 km |
| KF + classical + XNAV (1 hour) | ~123 m — **333× better** |
| KF + quantum + XNAV (1 hour) | ~0.015 m — **8 000× better** |

The quantum advantage comes from two sources:
- **Heisenberg-limited sensors**: 1/N scaling instead of 1/√N
- **Lower Kalman process noise**: the KF needs almost no external corrections

---
*QENS is open source. Contributions welcome.*
*GitHub: [github.com/georgizhlv/QENS](https://github.com/georgizhlv/QENS)*
